# 🔥 Advanced · VideoMAE on egocentric video

> 🔥 **Advanced / heavy lab.** Fine-tune the VideoMAE video transformer for first-person action recognition, with a confusion-matrix evaluation.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **30–60 min** including downloads. Built on the official **[transformers / VideoMAE](https://huggingface.co/docs/transformers/en/tasks/video_classification)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

Maps to lessons **C3–C4, C9**. Uses a small UCF subset out of the box and shows exactly how to point it at **EPIC-Kitchens-100 / Ego4D**.

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | T4 16 GB — videomae-base, small subset, fp16 | 4–8× A100 40 GB for EPIC-100 / Ego4D |
| **Storage** | UCF subset ~ 1 GB | EPIC-Kitchens-100 ~ 0.5–1 TB (RGB frames); Ego4D multi-TB |
| **Time** | 300–600 steps ~ 20–40 min | full fine-tune ~ 1–3 days on 4–8× A100 |

**Full pipeline (scale-up):** `torchrun --nproc_per_node=8 train.py …` over the full frame dataset, many epochs.

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi -L
!pip -q install "transformers>=4.41" evaluate pytorchvideo av scikit-learn

## 1 · Data
A ready-made UCF101 subset for the demo. **For egocentric**: arrange EPIC-Kitchens / Ego4D clips as `root/{train,val}/{action}/clip.mp4` and point `root` at it — everything below is unchanged.

In [ ]:
import os, tarfile
from huggingface_hub import hf_hub_download
arch = hf_hub_download(repo_id="sayakpaul/ucf101-subset", filename="UCF101_subset.tar.gz", repo_type="dataset")
if not os.path.exists("UCF101_subset"):
    with tarfile.open(arch) as t: t.extractall(".")
root = "UCF101_subset"   # <-- swap for your EPIC-Kitchens / Ego4D folder
labels = sorted({p.split("_")[1] for p in os.listdir(f"{root}/train")})
label2id = {l: i for i, l in enumerate(labels)}; id2label = {i: l for l, i in label2id.items()}
print(len(labels), "classes")

## 2 · Model + processor

In [ ]:
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification
ckpt = "MCG-NJU/videomae-base"
proc = VideoMAEImageProcessor.from_pretrained(ckpt)
model = VideoMAEForVideoClassification.from_pretrained(ckpt, label2id=label2id, id2label=id2label, ignore_mismatched_sizes=True)
nf = model.config.num_frames

## 3 · Pipelines (pytorchvideo) + Trainer

In [ ]:
import pytorchvideo.data, torch, numpy as np, evaluate
from pytorchvideo.transforms import ApplyTransformToKey, Normalize, UniformTemporalSubsample
from pytorchvideo.data import make_clip_sampler
from torchvision.transforms import Compose, Lambda, Resize, RandomCrop, CenterCrop
from transformers import TrainingArguments, Trainer
size = proc.size.get("shortest_edge", proc.size.get("height", 224)); dur = nf / 25
def tf(train):
    crop = RandomCrop(size) if train else CenterCrop(size)
    return ApplyTransformToKey("video", Compose([UniformTemporalSubsample(nf), Lambda(lambda x: x/255.0), Normalize(proc.image_mean, proc.image_std), Resize(size), crop]))
def ds(split, train):
    return pytorchvideo.data.Ucf101(os.path.join(root, split), make_clip_sampler("random" if train else "uniform", dur), decode_audio=False, transform=tf(train))
acc = evaluate.load("accuracy")
def collate(b): return {"pixel_values": torch.stack([e["video"].permute(1,0,2,3) for e in b]), "labels": torch.tensor([label2id[e["label"]] for e in b])}
args = TrainingArguments("vmae-ego", per_device_train_batch_size=4, per_device_eval_batch_size=4, learning_rate=5e-5,
    max_steps=600, eval_strategy="steps", eval_steps=200, logging_steps=25, warmup_ratio=0.1,
    fp16=True, remove_unused_columns=False, report_to="none")
trainer = Trainer(model, args, train_dataset=ds("train", True), eval_dataset=ds("val", False), data_collator=collate,
    compute_metrics=lambda p: acc.compute(predictions=np.argmax(p.predictions,1), references=p.label_ids))
trainer.train()

## 4 · Evaluate — confusion matrix

In [ ]:
import itertools, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
ys, ps = [], []
for b in [collate(list(itertools.islice(iter(ds("val", False)), 4))) for _ in range(8)]:
    with torch.no_grad():
        ps += model(b["pixel_values"].to(model.device)).logits.argmax(-1).cpu().tolist()
    ys += b["labels"].tolist()
cm = confusion_matrix(ys, ps, labels=list(range(len(labels))))
plt.imshow(cm); plt.title("confusion matrix"); plt.xlabel("pred"); plt.ylabel("true"); plt.colorbar(); plt.show()

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`. To publish any output folder as a **model repo** (then group repos into a **Collection** on your profile): `from huggingface_hub import HfApi; HfApi().upload_folder(folder_path="OUTPUT_DIR", repo_id="<you>/ropedia-<lab>")`.

## How this links to tracks A–D
Egocentric action models relate to:
- **A · Human** human motion / activity · **D · Scene / world** what happens where in a scene.

## Next steps
- **EPIC-Kitchens-100 / Ego4D**: register, download, arrange as folders above; raise `max_steps` and classes.
- `trainer.push_to_hub()` to save the model. Compare with the frozen-feature baseline (lesson C9).